In [65]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegressionCV
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from datetime import datetime
import warnings

## Description goes here

In [2]:
# load the training data
train = pd.read_parquet('data/training.parquet')
train.head()

,level_0,index,Month,bond_log_return,bond_price_max,bond_price_min,return_label,bond_max_min_return_diff,DATE,bond,...,Manufacturing_Production_log_return_discretized,Bitcoin_spot_price_log_return_sum,Bitcoin_spot_price_max,Bitcoin_spot_price_min,Bitcoin_spot_price_log_return_discretized,Bitcoin_spot_price_max_min_return_diff,Money_supply,Money_supply_log_return_sum,Money_supply_log_return_discretized,Term
0,0,0,2021-09-01,0.000350,99.918,99.883,Increase,0.000350,2021-09-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Decrease,-0.071135,52691.21,40656.14,Decrease,0.259299,20979.01,0.008040,Increase,2
1,1,1,2021-10-01,-0.003168,99.969,99.574,Decrease,0.003959,2021-10-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,0.335915,66005.18,47692.86,Increase,0.324952,21142.51,0.007763,Increase,2
2,2,2,2021-11-01,0.000040,99.781,99.402,Increase,0.003806,2021-11-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,-0.075391,67510.07,53839.40,Decrease,0.226271,21316.91,0.008215,Increase,2
3,3,3,2021-12-01,-0.003026,99.531,99.246,Decrease,0.002868,2021-12-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,-0.206989,57228.48,46240.16,Decrease,0.213203,21471.11,0.007208,Increase,2
25,25,0,2021-08-01,0.000782,99.844,99.758,Increase,0.000862,2021-08-01,United States Treasury Notes 0.125% 31-AUG-202...,...,Decrease,0.126516,49464.67,38207.28,Increase,0.258233,20811.01,0.008813,Increase,2


In [3]:
# convert the increase column to binary
discritized_vars = [x for x in train.columns if "_discretized" in x]
for col in discritized_vars:
    train[col] = train[col].apply(lambda x: 1 if x == "Increase" else 0)

train["return_label"] = train["return_label"].apply(lambda x: 1 if x == "Increase" else 0)
train.head()


,level_0,index,Month,bond_log_return,bond_price_max,bond_price_min,return_label,bond_max_min_return_diff,DATE,bond,...,Manufacturing_Production_log_return_discretized,Bitcoin_spot_price_log_return_sum,Bitcoin_spot_price_max,Bitcoin_spot_price_min,Bitcoin_spot_price_log_return_discretized,Bitcoin_spot_price_max_min_return_diff,Money_supply,Money_supply_log_return_sum,Money_supply_log_return_discretized,Term
0,0,0,2021-09-01,0.000350,99.918,99.883,1,0.000350,2021-09-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,0,-0.071135,52691.21,40656.14,0,0.259299,20979.01,0.008040,1,2
1,1,1,2021-10-01,-0.003168,99.969,99.574,0,0.003959,2021-10-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,0.335915,66005.18,47692.86,1,0.324952,21142.51,0.007763,1,2
2,2,2,2021-11-01,0.000040,99.781,99.402,1,0.003806,2021-11-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,-0.075391,67510.07,53839.40,0,0.226271,21316.91,0.008215,1,2
3,3,3,2021-12-01,-0.003026,99.531,99.246,0,0.002868,2021-12-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,-0.206989,57228.48,46240.16,0,0.213203,21471.11,0.007208,1,2
25,25,0,2021-08-01,0.000782,99.844,99.758,1,0.000862,2021-08-01,United States Treasury Notes 0.125% 31-AUG-202...,...,0,0.126516,49464.67,38207.28,1,0.258233,20811.01,0.008813,1,2


In [68]:
# create a function for the feature selection process
# def en_feature_selection(x: list[str], y: str, data: pd.DataFrame) -> list[str]:
#     """
#     Use Elastic Net to select the relevant features for the model
#     """
#     elastic = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, .99, 1],
#                            alphas=None,
#                            cv=None,
#                            random_state=None)
#     elastic.fit(data[x], data[y])
#     coef = pd.Series(elastic.coef_, index=x)
#     # return the selected features
#     return coef[coef != 0].index.tolist()

def fit_regression_model(x: list[str], y: str, data: pd.DataFrame):
    """
    Fit a regression model to the data
    """
    data = data.dropna(axis=0)
    log_regression = LogisticRegressionCV(
                            penalty='elasticnet',
                            solver='saga',
                            l1_ratios=[.1, .3, .5, .7, .9, .95, .99, 1])
    log_regression.fit(data[x], data[y])
    return log_regression


def get_historical_data(data: pd.DataFrame, x: list[str], y: str, time_window: int, term: int, current_date: date) -> pd.DataFrame:
    """
    Get the historical data for the given time window
    """
    start_date = (current_date - relativedelta(months= 1)) - relativedelta(months=time_window)
    full_columns = x + [y]
    return data[(data['Month'] >= start_date) & (data['Month'] < current_date) & (data['term'] == term)][full_columns]

def get_prediction(model: LogisticRegressionCV, row_vals: pd.Series, actuals: pd.Series) -> pd.DataFrame:
    """
    Get the prediction for the given row
    """

    predictions = model.predict(row_vals)
    probs = model.predict_proba(row_vals)
    df_results = pd.DataFrame(probs, columns=['Positive', 'Negative'])
    df_results['prediction'] = predictions
    df_results['actual'] = actuals.values
    row_test.reset_index(drop=True, inplace=True)
    return pd.concat([row_test, df_results], axis=1)



def split_data(data: pd.DataFrame, split: float = 0.75) -> pd.DataFrame:
    """
    Split the data into training and testing sets
    """
    split_index = int(len(data) * split)
    # train_data = data[:split_index]
    data.sort_values(by='Month', ascending=True, inplace=True)
    validation_data = data[split_index:]
    return validation_data


def run_end_to_end_model(
        train_data: pd.DataFrame,
        x: list[str],
        y: str,
        time_window: int,
        split: float = 0.75) -> tuple:
    """
    Run the end to end model
    """
    # split the data into training and validation sets
    dfs = []
    df_tmp = train_data.copy()
    validation_data = split_data(data=train_data, split=split)
    test_min_date = validation_data['Month'].min()
    print(f"Validation data starts with {test_min_date}... running model...")
    full_columns = x + [y]
    for group, group_df in tqdm(validation_data.groupby(['Month', 'term'])):
    # drop any columns that are null
        df_modeling_data = group_df[full_columns].dropna(axis=1)
        feature_columns = [x for x in df_modeling_data.columns if x != y]
        historical_data = get_historical_data(
                            data=df_tmp,
                            x=feature_columns,
                            y=y,
                            time_window=time_window,
                            term=group[1],
                            current_date=group[0]
            )
        regression_model = fit_regression_model(x=feature_columns, y=y, data=historical_data)
        row_vals = group_df.iloc[:][x]
        prediction = get_prediction(model=regression_model, row_vals=row_vals, actuals=group_df[y])
        dfs.append(prediction)
    return pd.concat(dfs)



In [22]:
# train.head()
test_function = get_historical_data(
                        data=train,
                        x=['Month', 'bond', 'term', '5_Year_Inflation_Expectation_log_return_sum', '10_Year_Inflation_Expectation_log_return_discretized', '10_Year_3_Year_Spread_max_min_return_diff'],
                        y='return_label',
                        time_window=12,
                        term=2,
                        current_date=datetime(2021, 12, 1, 0, 0, 0)
        )

test_function


,Month,bond,term,5_Year_Inflation_Expectation_log_return_sum,10_Year_Inflation_Expectation_log_return_discretized,10_Year_3_Year_Spread_max_min_return_diff,return_label
0,2021-09-01,United States Treasury Notes 0.25% 30-SEP-2023...,2,0.011976,1,0.195567,1
1,2021-10-01,United States Treasury Notes 0.25% 30-SEP-2023...,2,0.140452,1,0.117016,0
2,2021-11-01,United States Treasury Notes 0.25% 30-SEP-2023...,2,-0.031526,0,0.153122,1
25,2021-08-01,United States Treasury Notes 0.125% 31-AUG-202...,2,-0.031623,0,0.137870,1
26,2021-09-01,United States Treasury Notes 0.125% 31-AUG-202...,2,0.011976,1,0.195567,0
...,...,...,...,...,...,...,...
5945,2020-12-01,United States Treasury Notes 2.5% 31-JAN-2021 ...,2,0.148216,1,0.081917,0
5946,2021-01-01,United States Treasury Notes 2.5% 31-JAN-2021 ...,2,0.124563,1,0.239480,0
5970,2020-11-01,United States Treasury Notes 2.5% 31-DEC-2020 ...,2,0.054725,1,0.254530,0
5971,2020-12-01,United States Treasury Notes 2.5% 31-DEC-2020 ...,2,0.148216,1,0.081917,0


In [21]:
print(test_function['Month'].min())
print(test_function['Month'].max())

print(test_function['term'].unique())

2020-11-01 00:00:00
2021-11-01 00:00:00
[2]


In [38]:
# test the regression model
regression_model_test = fit_regression_model(x=['5_Year_Inflation_Expectation_log_return_sum', '10_Year_Inflation_Expectation_log_return_discretized', '10_Year_3_Year_Spread_max_min_return_diff'], y='return_label', data=test_function)

row_test = test_function.iloc[0:5][['5_Year_Inflation_Expectation_log_return_sum', '10_Year_Inflation_Expectation_log_return_discretized', '10_Year_3_Year_Spread_max_min_return_diff']]

predictions, probs = get_prediction(model=regression_model_test, row_vals=row_test)

In [49]:
df_results = pd.DataFrame(probs, columns=['Positive', 'Negative'])
df_results['prediction'] = predictions
row_test.reset_index(drop=True, inplace=True)
final_results = pd.concat([row_test, df_results], axis=1)
final_results

,5_Year_Inflation_Expectation_log_return_sum,10_Year_Inflation_Expectation_log_return_discretized,10_Year_3_Year_Spread_max_min_return_diff,Positive,Negative,prediction
0,0.011976,1,0.195567,0.7039,0.2961,0
1,0.140452,1,0.117016,0.7039,0.2961,0
2,-0.031526,0,0.153122,0.7039,0.2961,0
3,-0.031623,0,0.137870,0.7039,0.2961,0
4,0.011976,1,0.195567,0.7039,0.2961,0


In [69]:
# turn off warnings
warnings.filterwarnings('ignore')
df_prediction_results = run_end_to_end_model(train_data=train,
                                             x=['5_Year_Inflation_Expectation_log_return_sum', '10_Year_Inflation_Expectation_log_return_discretized', '10_Year_3_Year_Spread_max_min_return_diff'],
                                             y='return_label',
                                             time_window=36,
                                             split=0.95)

Validation data starts with 2021-06-01 00:00:00... running model...


100%|██████████| 28/28 [00:14<00:00,  1.90it/s]


In [70]:
df_prediction_results

,5_Year_Inflation_Expectation_log_return_sum,10_Year_Inflation_Expectation_log_return_discretized,10_Year_3_Year_Spread_max_min_return_diff,Positive,Negative,prediction,actual
0,0.011976,1.0,0.195567,0.395339,0.604661,1.0,0.0
1,0.140452,1.0,0.117016,0.395339,0.604661,1.0,0.0
2,-0.031526,0.0,0.153122,0.395339,0.604661,1.0,1.0
3,-0.031623,0.0,0.137870,0.395339,0.604661,1.0,1.0
4,0.011976,1.0,0.195567,0.395339,0.604661,1.0,0.0
...,...,...,...,...,...,...,...
63,NaN,NaN,NaN,0.514571,0.485429,0.0,0.0
64,NaN,NaN,NaN,0.514571,0.485429,0.0,0.0
65,NaN,NaN,NaN,0.514571,0.485429,0.0,0.0
66,NaN,NaN,NaN,0.514571,0.485429,0.0,0.0
